# RecA-TB PaDEL Fingerprint and Modeling Matrix Workflow

Notebook ini menggabungkan dua script:

1. `02_Calculating_molecular_fingerprints_using_PaDEL.py`
2. `02a_Fingerprint_dataset_matrix_assembly.py`

Tujuannya adalah menghitung molecular fingerprints menggunakan PaDEL, lalu menyusun dataset final siap machine learning untuk studi QSAR klasifikasi RecA-TB.


## Workflow Summary

**Input utama:**
- `outputs/recA_chembl/recA_ec50_binary.csv`
- `outputs/01_recA_ec50_binary_balanced_50_50.csv`
- `outputs/recA_chembl/padel/combined_fingerprints.csv`

**Output utama:**
- `outputs/recA_chembl/padel/combined_fingerprints.csv`
- `outputs/02_recA_modeling_matrix.csv`
- `outputs/02_fingerprint_dataset_summary.csv`


## Important Note

PaDEL membutuhkan Java dan package `padelpy`. Jika belum terpasang, jalankan:

```bash
pip install padelpy
```

Pastikan juga dataset hasil data curation sebelumnya sudah tersedia sebelum menjalankan notebook ini.


## Part 1 — Calculate PaDEL Molecular Fingerprints

In [ ]:
from __future__ import annotations

import argparse
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import padelpy
from padelpy import padeldescriptor


SCRIPT_DIR = Path(__file__).resolve().parent

INPUT_FILE = SCRIPT_DIR / "outputs" / "recA_chembl" / "recA_ec50_binary.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "recA_chembl" / "padel"

SMILES_FILE = OUTPUT_DIR / "molecule.smi"
SUBSET_FILE = OUTPUT_DIR / "recA_binary_padel_input.csv"
COMBINED_FILE = OUTPUT_DIR / "combined_fingerprints.csv"

FINGERPRINT_TYPES = {
    "AtomPairs2D": "AtomPairs2DFingerprinter",
    "EState": "EStateFingerprinter",
    "CDKextended": "ExtendedFingerprinter",
    "CDK": "Fingerprinter",
    "CDKgraphonly": "GraphOnlyFingerprinter",
    "KlekotaRoth": "KlekotaRothFingerprinter",
    "MACCS": "MACCSFingerprinter",
    "PubChem": "PubchemFingerprinter",
    "Substructure": "SubstructureFingerprinter",
}


def load_dataset(limit: int | None = None) -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run the data curation script first to generate recA_ec50_binary.csv."
        )

    df = pd.read_csv(INPUT_FILE)

    expected = {
        "molecule_chembl_id",
        "canonical_smiles",
        "bioactivity_class",
        "bioactivity_numeric",
    }

    missing = expected.difference(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {sorted(missing)}")

    df = df.dropna(subset=["molecule_chembl_id", "canonical_smiles"]).copy()

    if limit is not None:
        df = df.head(limit).copy()

    return df


def prepare_subset(df: pd.DataFrame) -> pd.DataFrame:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    subset = df[
        [
            "molecule_chembl_id",
            "canonical_smiles",
            "bioactivity_class",
            "bioactivity_numeric",
        ]
    ].copy()

    subset = subset.rename(columns={"bioactivity_numeric": "class"})
    subset.to_csv(SUBSET_FILE, index=False)

    return subset


def write_smiles_file(subset: pd.DataFrame) -> None:
    lines = []

    for smiles, name in subset[["canonical_smiles", "molecule_chembl_id"]].itertuples(index=False):
        lines.append(f"{smiles}\t{name}")

    SMILES_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")


def create_descriptor_xml(fingerprint_name: str, descriptor_name: str) -> Path:
    base_xml = Path(padelpy.__file__).parent / "PaDEL-Descriptor" / "descriptors.xml"

    if not base_xml.exists():
        raise FileNotFoundError(f"PaDEL descriptors.xml not found:\n{base_xml}")

    output_xml = OUTPUT_DIR / f"{fingerprint_name}.xml"

    tree = ET.parse(base_xml)
    root = tree.getroot()

    found = False

    for descriptor in root.iter("Descriptor"):
        is_target = descriptor.attrib.get("name") == descriptor_name
        descriptor.set("value", "true" if is_target else "false")

        if is_target:
            found = True

    if not found:
        raise ValueError(f"Descriptor {descriptor_name} not found in {base_xml}")

    tree.write(output_xml, encoding="utf-8", xml_declaration=False)

    return output_xml


def calculate_fingerprints(threads: int, timeout: int | None) -> pd.DataFrame:
    all_descriptors: pd.DataFrame | None = None

    for fingerprint, descriptor_name in FINGERPRINT_TYPES.items():
        print(f"\nCalculating {fingerprint} fingerprints...")

        descriptor_xml = create_descriptor_xml(fingerprint, descriptor_name)
        fingerprint_output_file = OUTPUT_DIR / f"{fingerprint}.csv"

        padeldescriptor(
            mol_dir=str(SMILES_FILE.resolve()),
            d_file=str(fingerprint_output_file.resolve()),
            descriptortypes=str(descriptor_xml.resolve()),
            detectaromaticity=True,
            standardizenitro=True,
            standardizetautomers=True,
            removesalt=True,
            log=True,
            fingerprints=True,
            threads=threads,
            sp_timeout=timeout,
        )

        if not fingerprint_output_file.exists():
            raise RuntimeError(f"PaDEL failed to generate: {fingerprint_output_file}")

        descriptors = pd.read_csv(fingerprint_output_file)

        if "Name" not in descriptors.columns:
            raise ValueError(f"'Name' column not found in {fingerprint_output_file}")

        if all_descriptors is None:
            all_descriptors = descriptors
        else:
            all_descriptors = all_descriptors.merge(
                descriptors,
                on="Name",
                how="inner",
            )

    if all_descriptors is None:
        raise RuntimeError("No fingerprint tables were generated.")

    return all_descriptors


def build_combined_dataset(
    all_descriptors: pd.DataFrame,
    subset: pd.DataFrame,
) -> pd.DataFrame:
    labels = subset.rename(columns={"molecule_chembl_id": "Name"})

    combined = all_descriptors.merge(
        labels,
        on="Name",
        how="inner",
    )

    combined.to_csv(COMBINED_FILE, index=False)

    return combined

In [ ]:
# ============================================================
# NOTEBOOK RUNNER: PaDEL fingerprint calculation
# ============================================================

# Optional quick test:
# LIMIT = 10
LIMIT = None

THREADS = 2
TIMEOUT = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = load_dataset(limit=LIMIT)
subset = prepare_subset(df)
write_smiles_file(subset)

all_descriptors = calculate_fingerprints(
    threads=THREADS,
    timeout=TIMEOUT,
)

combined = build_combined_dataset(all_descriptors, subset)

print("\nPaDEL fingerprint workflow completed successfully.")
print(f"Input rows: {len(df)}")
print(f"Subset rows: {len(subset)}")
print(f"Fingerprint table shape: {all_descriptors.shape}")
print(f"Combined dataset shape: {combined.shape}")

print("\nClass distribution:")
print(combined["bioactivity_class"].value_counts().to_string())

print(f"\nCombined output saved to:\n{COMBINED_FILE}")

## Part 2 — Assemble Final Fingerprint Modeling Matrix

In [ ]:
"""PaDEL fingerprint matrix assembly for the curated RecA-TB dataset.

Methodological basis:
- PaDEL-Descriptor / PaDEL fingerprints:
  Yap, Journal of Computational Chemistry (2011),
  doi:10.1002/jcc.21707

- ChEMBL curated activity source:
  Zdrazil et al., Nucleic Acids Research (2024),
  doi:10.1093/nar/gkad1004

This script merges:
1. Curated balanced RecA binary bioactivity dataset
2. Precomputed PaDEL fingerprint matrix

The final output is a machine-learning-ready modeling matrix
for QSAR classification studies.
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd


# ============================================================
# PATH CONFIGURATION
# ============================================================

SCRIPT_DIR = Path(__file__).resolve().parent

OUTPUT_DIR = SCRIPT_DIR / "outputs"

CURATED_BINARY_FILE = (
    OUTPUT_DIR / "01_recA_ec50_binary_balanced_50_50.csv"
)

PREFINGERPRINT_FILE = (
    SCRIPT_DIR
    / "outputs"
    / "recA_chembl"
    / "padel"
    / "combined_fingerprints.csv"
)


# ============================================================
# LOAD DATASETS
# ============================================================

def load_datasets() -> tuple[pd.DataFrame, pd.DataFrame]:

    if not CURATED_BINARY_FILE.exists():
        raise FileNotFoundError(
            f"\nCurated binary dataset not found:\n"
            f"{CURATED_BINARY_FILE}\n\n"
            "Please run the data curation workflow first."
        )

    if not PREFINGERPRINT_FILE.exists():
        raise FileNotFoundError(
            f"\nFingerprint dataset not found:\n"
            f"{PREFINGERPRINT_FILE}\n\n"
            "Please run the PaDEL fingerprint workflow first."
        )

    binary = pd.read_csv(CURATED_BINARY_FILE)
    fp = pd.read_csv(PREFINGERPRINT_FILE)

    return binary, fp


# ============================================================
# BUILD MODELING MATRIX
# ============================================================

def build_modeling_matrix(
    binary: pd.DataFrame,
    fp: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:

    fp_meta = {
        "Name",
        "canonical_smiles",
        "bioactivity_class",
        "class",
        "bioactivity_numeric",
        "standard_value",
    }

    fp_feature_cols = [
        c for c in fp.columns
        if c not in fp_meta
    ]

    if "Name" not in fp.columns:
        raise ValueError(
            "'Name' column not found in fingerprint dataset."
        )

    if "molecule_chembl_id" not in binary.columns:
        raise ValueError(
            "'molecule_chembl_id' column not found in curated dataset."
        )

    modeling = binary.merge(
        fp[["Name", *fp_feature_cols]],
        left_on="molecule_chembl_id",
        right_on="Name",
        how="inner",
        validate="one_to_one",
    )

    if modeling.empty:
        raise ValueError(
            "Fingerprint merge failed. No matching compounds found."
        )

    if len(modeling) != len(binary):
        raise ValueError(
            f"\nFingerprint merge mismatch:\n"
            f"Binary dataset rows = {len(binary)}\n"
            f"Merged dataset rows = {len(modeling)}"
        )

    final = pd.concat(
        [
            modeling[
                [
                    "Name",
                    "canonical_smiles",
                    "bioactivity_class",
                    "class",
                ]
            ].copy(),

            modeling[fp_feature_cols].copy(),
        ],
        axis=1,
    )

    return final, fp_feature_cols


# ============================================================
# SAVE OUTPUTS
# ============================================================

def save_outputs(
    final: pd.DataFrame,
    fp_feature_cols: list[str],
) -> None:

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    modeling_output = (
        OUTPUT_DIR / "02_recA_modeling_matrix.csv"
    )

    summary_output = (
        OUTPUT_DIR / "02_fingerprint_dataset_summary.csv"
    )

    final.to_csv(modeling_output, index=False)

    summary = pd.DataFrame(
        [
            {
                "rows": len(final),
                "total_columns": final.shape[1],
                "feature_columns": len(fp_feature_cols),
                "active_count": int(
                    (final["bioactivity_class"] == "active").sum()
                ),
                "inactive_count": int(
                    (final["bioactivity_class"] == "inactive").sum()
                ),
            }
        ]
    )

    summary.to_csv(summary_output, index=False)

    print("\nModeling matrix generated successfully.")
    print(f"\nRows: {len(final)}")
    print(f"Total columns: {final.shape[1]}")
    print(f"Fingerprint features: {len(fp_feature_cols)}")

    print("\nClass distribution:")
    print(
        final["bioactivity_class"]
        .value_counts()
        .to_string()
    )

    print(f"\nModeling matrix saved to:\n{modeling_output}")
    print(f"\nSummary saved to:\n{summary_output}")


# ============================================================
# MAIN
# ============================================================

In [ ]:
# ============================================================
# NOTEBOOK RUNNER: Build final modeling matrix
# ============================================================

binary, fp = load_datasets()

final, fp_feature_cols = build_modeling_matrix(
    binary,
    fp,
)

save_outputs(
    final,
    fp_feature_cols,
)

## Final Outputs

Setelah semua cell berhasil dijalankan, file utama untuk tahap machine learning adalah:

```text
outputs/02_recA_modeling_matrix.csv
```

File ini berisi:
- `Name`
- `canonical_smiles`
- `bioactivity_class`
- `class`
- semua fitur fingerprint PaDEL
